## Data Augmentation

In [11]:
import pandas as pd
import re 
import nltk
import numpy as np
import nlpaug.augmenter.word as naw
from io import StringIO

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Azeroual\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
#Chargement de dataset 

df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)
df.to_csv("SMSSpamAugment.csv", index=False)
df = pd.read_csv("SMSSpamAugment.csv")

## <span style="color:#A23B72">Partie 0 — Augmentation des donnees</span>

In [ ]:
print(f"📊 Taille avant augmentation : {len(df)} lignes")

# 2. Configuration de l'augmentation
# On utilise le remplacement par synonymes (ex: "win" -> "gain")
# aug = naw.SynonymAug(aug_src='wordnet') # Nécessite téléchargement NLTK
# Pour faire simple et rapide sans config NLTK, on utilise l'insertion/suppression aléatoire :
aug = naw.RandomWordAug(action='substitute', aug_min=1, aug_max=3) 

# 3. Fonction pour augmenter les données
def augment_data(df, label_col='label', text_col='message', n_augments=3):
    new_rows = []
    
    for index, row in df.iterrows():
        original_text = row[text_col]
        label = row[label_col]
        
        # On garde l'original
        new_rows.append(row)
        
        # On crée des variantes
        for _ in range(n_augments):
            try:
                # Augmentation du texte
                augmented_text = aug.augment(original_text)
                
                # Création d'une nouvelle ligne avec le même label
                new_row = row.copy()
                new_row[text_col] = augmented_text
                new_rows.append(new_row)
            except:
                # Si l'augmentation échoue (texte trop court), on garde l'original
                continue
                
    return pd.DataFrame(new_rows)

# 4. Application
df_augmented = augment_data(df, n_augments=2) # 2 variantes par message

print(f"🚀 Taille après augmentation : {len(df_augmented)} lignes")
print("\n--- Aperçu des données augmentées ---")
print(df_augmented[['label', 'message']].to_string())

# 5. Sauvegarder si besoin
df_augmented.to_csv('sms_spam_augmented.csv', index=False)

## <span style="color:#A23B72">Partie 1 — Nettoyage des Données</span>

In [6]:
#Mise en minuscules
df['message'] = df['message'].str.lower()
#Suppression de la ponctuation
df['message'] = df['message'].str.replace(r"[^\w\s]","",regex=True)
#Suppression des chiffres
df['message'] = df['message'].str.replace(r"[\d]", "",regex=True)
#Suppression des caractères spéciaux
df['message'] = df['message'].str.replace(r"[@#$%^&*()_+=<>]","", regex=True)
#Supprimer répétition excessive des lettres

def remove_duplicate_charactere(text) :
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

#Supprimer les mots doublés
def remove_duplicate_words(text) :
    words = text.split()
    unique_words = list(dict.fromkeys(words))
    return " ".join(unique_words)
df['message'] = df['message'].apply(remove_duplicate_words)
df['message'] = df['message'].apply(remove_duplicate_charactere)

df.to_csv("SMSSpamAugment.csv", index=False)


## <span style="color:#A23B72">Partie 2 — Prétraitement Linguistique</span>

In [7]:
def remove_stopwords(words , stop_words):
    sans_stpwords = []
    for word in words :
        if word not in stop_words and len(word) >= 2:
            sans_stpwords.append(word)
    return sans_stpwords

def apply_Stemming(words,stemmer):
    return [stemmer.stem(word) for word in words]

def apply_lemmatization(words,lemmatizer):
    return [lemmatizer.lemmatize(word) for word in words]